In [ ]:
from google.colab import drive
import os

# 1. Conectar a Google Drive
drive.mount('/content/drive')

# 2. Tu ruta exacta
DATA_DIR = '/content/drive/MyDrive/Modelos/maize_dataset/clean'

# 3. Verificación de las carpetas
if os.path.exists(DATA_DIR):
    clases = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f" ¡Ruta encontrada! Clases detectadas listas para entrenar: {clases}")
else:
    print(f" Error: No se encontró la ruta {DATA_DIR}. Verifica que el nombre sea exacto.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 ¡Ruta encontrada! Clases detectadas listas para entrenar: ['potassium_deficiency', 'fall_armyworm', 'phosphorus_deficiency', 'healthy', 'common_rust', 'northern_corn_leaf_blight', 'aphids_pest', 'gray_leaf_spot', 'nitrogen_deficiency']


In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models, applications

# Verificar que Colab te asignó una GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f" ¡Excelente! GPU detectada: {gpus}")
else:
    print("ADVERTENCIA: No se detectó GPU. Ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno de ejecución' y selecciona 'T4 GPU'.")

 ¡Excelente! GPU detectada: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
BASE_DIR = '/content/drive/MyDrive/Modelos/maize_dataset/clean'

print("Cargando y dividiendo el dataset automáticamente...")

train_dataset_raw = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_dataset_raw.class_names

val_test_dataset = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_batches = tf.data.experimental.cardinality(val_test_dataset)
val_dataset = val_test_dataset.take(val_batches // 2)
test_dataset = val_test_dataset.skip(val_batches // 2)

# ¡AQUÍ ESTÁ LA MAGIA SALVA-RAM!
# Quitamos el .cache() y bajamos el shuffle a 300
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset_raw.shuffle(300).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

print(f"\n✅ Conjuntos de datos listos. Clases detectadas ({len(class_names)}): {class_names}")

Cargando y dividiendo el dataset automáticamente...
Found 25284 files belonging to 9 classes.
Using 20228 files for training.
Found 25284 files belonging to 9 classes.
Using 5056 files for validation.

✅ Conjuntos de datos listos. Clases detectadas (9): ['aphids_pest', 'common_rust', 'fall_armyworm', 'gray_leaf_spot', 'healthy', 'nitrogen_deficiency', 'northern_corn_leaf_blight', 'phosphorus_deficiency', 'potassium_deficiency']


In [ ]:
print(f"Clases a balancear: {class_names}")
print("Calculando desbalance de forma optimizada en memoria...")

# Truco: Extraemos solo las etiquetas (y) sin procesar los tensores de imágenes (x)
y_train_indices = np.concatenate([np.argmax(y.numpy(), axis=1) for x, y in train_dataset_raw], axis=0)

# Calculamos los pesos
pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_indices),
    y=y_train_indices
)
class_weights_dict = dict(enumerate(pesos))

print("\nPesos de compensación generados:")
for idx, clase in enumerate(class_names):
    print(f" - {clase}: {class_weights_dict[idx]:.4f}")

Clases a balancear: ['aphids_pest', 'common_rust', 'fall_armyworm', 'gray_leaf_spot', 'healthy', 'nitrogen_deficiency', 'northern_corn_leaf_blight', 'phosphorus_deficiency', 'potassium_deficiency']
Calculando desbalance de forma optimizada en memoria...

Pesos de compensación generados:
 - aphids_pest: 38.7510
 - common_rust: 1.2175
 - fall_armyworm: 0.5799
 - gray_leaf_spot: 2.5140
 - healthy: 0.3199
 - nitrogen_deficiency: 5.5771
 - northern_corn_leaf_blight: 0.4146
 - phosphorus_deficiency: 4.5497
 - potassium_deficiency: 10.6519


In [ ]:
# Capas de Data Augmentation
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2),
  layers.RandomContrast(0.1),
], name="data_augmentation_layer")

# Cargar preentrenado (MobileNetV3Small para dispositivos móviles)
base_model = applications.MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=True
)
base_model.trainable = False # Congelamos el conocimiento base

# Ensamblamos la red final
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

# ¡AQUÍ ESTÁ LA MAGIA! El número de neuronas se adapta automáticamente a las carpetas encontradas
num_clases = len(class_names)
outputs = layers.Dense(num_clases, activation='softmax', name='prediccion_enfermedad')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

print(f" Arquitectura construida para {num_clases} clases. Resumen del modelo:")
model.summary()

 Arquitectura construida para 9 clases. Resumen del modelo:


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation_layer         │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prediccion_enfermedad (Dense)   │ (None, 9)              │         5,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 944,313 (3.60 MB)

 Trainable params: 5,193 (20.29 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [ ]:
EPOCHS = 20

# Ruta donde se guardará el modelo entrenado de vuelta a tu Google Drive
MODEL_SAVE_PATH = '/content/drive/MyDrive/DoctorMaiz_Project/mobilenetv3_maiz_mejor_modelo.keras'

# Callback: Guarda automáticamente el modelo si mejora la pérdida de validación
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    MODEL_SAVE_PATH,
    save_best_only=True,
    monitor='val_loss',
    mode='min',
    verbose=1
)

# Callback: Detiene el entrenamiento temprano si el modelo deja de aprender
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True
)

print("\n ¡Iniciando el entrenamiento en la GPU!")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    class_weight=class_weights_dict,
    callbacks=[checkpoint_cb, early_stopping_cb]
)

print(f"\n Entrenamiento finalizado. El mejor modelo ha sido guardado de forma segura en tu Drive: {MODEL_SAVE_PATH}")


 ¡Iniciando el entrenamiento en la GPU!
Epoch 1/20
632/633 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step - accuracy: 0.4900 - loss: 1.5455 - precision: 0.7490 - recall: 0.2664
Epoch 1: val_loss improved from None to 0.72870, saving model to /content/drive/MyDrive/DoctorMaiz_Project/mobilenetv3_maiz_mejor_modelo.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DoctorMaiz_Project/mobilenetv3_maiz_mejor_modelo.keras
633/633 ━━━━━━━━━━━━━━━━━━━━ 1376s 2s/step - accuracy: 0.6180 - loss: 1.1578 - precision: 0.8294 - recall: 0.4188 - val_accuracy: 0.7575 - val_loss: 0.7287 - val_precision: 0.8676 - val_recall: 0.6404
Epoch 2/20
632/633 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.7407 - loss: 0.7581 - precision: 0.8443 - recall: 0.6081 
Epoch 2: val_loss improved from 0.72870 to 0.61162, saving model to /content/drive/MyDrive/DoctorMaiz_Project/mobilenetv3_maiz_mejor_modelo.keras

Epoch 2: finished saving model to /content/drive/MyDrive/DoctorMaiz_Project/mobilenetv3_maiz_mejor_modelo.k

In [ ]:
import matplotlib.pyplot as plt

# Extraer datos del historial de entrenamiento
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

# Gráfica de Precisión (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Precisión Entrenamiento', marker='o')
plt.plot(epochs_range, val_acc, label='Precisión Validación', marker='o')
plt.legend(loc='lower right')
plt.title('Precisión de Entrenamiento y Validación')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.grid(True, linestyle='--', alpha=0.6)

# Gráfica de Pérdida (Loss)
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Pérdida Entrenamiento', marker='o')
plt.plot(epochs_range, val_loss, label='Pérdida Validación', marker='o')
plt.legend(loc='upper right')
plt.title('Pérdida de Entrenamiento y Validación')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("Evaluando el modelo con el conjunto de prueba (Test Dataset)...")
# 1. Obtener las métricas generales
test_loss, test_acc, test_prec, test_rec = model.evaluate(test_dataset)
print(f"\nResultados Finales en Test -> Precisión: {test_acc*100:.2f}% | Pérdida: {test_loss:.4f}")

# 2. Extraer predicciones y etiquetas reales para la Matriz
y_true = []
y_pred = []

print("Generando predicciones detalladas...")
for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

# 3. Imprimir el Reporte de Clasificación (Aquí verás el Macro F1-Score)
print("\n" + "="*50)
print("REPORTE DE CLASIFICACIÓN FINAL")
print("="*50)
print(classification_report(y_true, y_pred, target_names=class_names))

# 4. Dibujar la Matriz de Confusión
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de Confusión en Conjunto de Prueba')
plt.xlabel('Predicción del Modelo')
plt.ylabel('Etiqueta Real')
plt.show()

In [ ]:
import tensorflow as tf

print("Iniciando conversión a TensorFlow Lite...")

# 1. Cargar el convertidor con tu modelo entrenado
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 2. Aplicar optimización dinámica (Cuantización)
# Esto reduce drásticamente el peso del archivo sacrificando casi nada de precisión
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 3. Realizar la conversión
tflite_model = converter.convert()

# 4. Guardar el modelo .tflite en tu Drive
tflite_path = '/content/drive/MyDrive/Modelos/dataset-corn/doctor_maiz_optimizado.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

# 5. Guardar un archivo .txt con los nombres de las clases en orden
# (Súper importante para que Vue sepa qué significa la predicción 0, 1 o 2)
labels_path = '/content/drive/MyDrive/Modelos/dataset-corn/etiquetas_maiz.txt'
with open(labels_path, 'w') as f:
    f.write('\n'.join(class_names))

print(f"\n ¡Éxito! Modelo TFLite comprimido guardado en: {tflite_path}")
print(f" Etiquetas guardadas en: {labels_path}")